In [ ]:
import numpy as np

loaded = np.load("./datacache/sampled_features.npz")
sampled_features = {key: loaded[key] for key in loaded.files}
for (key, value) in sampled_features.items():
    print(key, value.shape)

In [ ]:
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

num_tasks = 4
num_categories = 5
final_points: dict[str, np.ndarray] = {}
for t in range(num_tasks):
    task_plain: list[np.ndarray] = []
    interval: dict[str, tuple[int, int]] = {}
    cnt = 0
    for c in range(num_categories):
        def append_to_task_plain(feature_key: str, cnt: int):
            feature = sampled_features[feature_key]
            task_plain.append(feature)
            interval[feature_key] = (cnt, cnt + feature.shape[0])
            cnt += feature.shape[0]
            return cnt
        cnt = append_to_task_plain(f'task{t}_class{c}_feature', cnt)
        cnt = append_to_task_plain(f'task{t}_class{c}_gen_feature', cnt)
        cnt = append_to_task_plain(f'task{t}_class{c}_untrained_gen_feature', cnt)
    task_plain = np.concatenate(task_plain, axis=0)
    pca = PCA(n_components=50, random_state=42)
    tsne = TSNE(n_components=2, random_state=42)
    pca_task_plain = pca.fit_transform(task_plain)
    tsne_task_plain = tsne.fit_transform(pca_task_plain)
    for (key, value) in interval.items():
        final_points[key] = tsne_task_plain[interval[key][0]: interval[key][1]]
for (key, value) in final_points.items():
    print(key, value)

In [ ]:
np.savez('./datacache/final_points.npz', **final_points)